In [2]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate, RepeatedStratifiedKFold, RandomizedSearchCV
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from mpl_toolkits.mplot3d import Axes3D
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from scipy.stats import randint, loguniform
from matplotlib.ticker import FuncFormatter
from sklearn.metrics import confusion_matrix
from matplotlib.colors import LinearSegmentedColormap
from scipy.spatial.distance import pdist, squareform

Duomenų paruošimas

In [3]:
data = pd.read_csv('duomenys.csv', encoding='utf-8') #Hunt katalogas
df = data[(data['Prob'] > 0.9) & (data["Plx"] > 0.3)]
cluster_size = df['Name'].value_counts()
large_clusters = cluster_size[cluster_size >= 20].index
df = df[df['Name'].isin(large_clusters)]

In [4]:
def prep_5d_astrometry(df): #koordinačių sistemos paruošimas
    cluster_results = []
    metadata = {}
    for cluster_name, c_data in df.groupby("Name"):
        c_data = c_data.copy()
        cols_needed = ["RA_ICRS", "DE_ICRS", "Plx", "pmRA", "pmDE", "e_pmRA", "e_pmDE",]
        c_data = c_data.dropna(subset=cols_needed)

        if len(c_data) < 20:
            continue

        med_dist = 1000.0 / np.median(c_data["Plx"])
        ra_rad = np.deg2rad(c_data["RA_ICRS"].values)
        dec_rad = np.deg2rad(c_data["DE_ICRS"].values)

        u_stars = np.column_stack([
            np.cos(dec_rad) * np.cos(ra_rad),
            np.cos(dec_rad) * np.sin(ra_rad),
            np.sin(dec_rad)
        ])

        u_los = u_stars.mean(axis=0)
        u_los = u_los / np.linalg.norm(u_los)

        C = u_los * med_dist

        if abs(u_los[2]) < 0.99:
            tmp = np.array([0.0, 0.0, 1.0])
        else:
            tmp = np.array([1.0, 0.0, 0.0])

        e1 = tmp - np.dot(tmp, u_los) * u_los
        e1 = e1 / np.linalg.norm(e1)
        e2 = np.cross(u_los, e1)

        distances = 1000.0 / c_data["Plx"].values
        r_vecs = u_stars * distances[:, None] - C
        r_meds = u_stars * med_dist - C
        X_perp = r_meds @ e1
        Y_perp = r_meds @ e2
        Z_los = r_vecs @ u_los

        D_trans = np.sqrt((np.var(X_perp) + np.var(Y_perp)) / 2.0)

        if D_trans <= 0:
            D_trans = 1.0

        var_pm_total = (np.var(c_data["pmRA"]) + np.var(c_data["pmDE"]))
        var_pm_error = (np.median(c_data["e_pmRA"])**2 + np.median(c_data["e_pmDE"])**2)

        sigma_pm_intrinsic = np.sqrt(max(var_pm_total * 0.25, var_pm_total - var_pm_error))

        if sigma_pm_intrinsic <= 0:
            sigma_pm_intrinsic = 1.0

        c_data["X_perp"] = X_perp
        c_data["Y_perp"] = Y_perp
        c_data["Z_los"] = Z_los

        c_data["X_norm"] = X_perp / D_trans
        c_data["Y_norm"] = Y_perp / D_trans
        c_data["Z_norm"] = Z_los / D_trans

        c_data["pmRA_norm"] = (c_data["pmRA"] - np.median(c_data["pmRA"])) / sigma_pm_intrinsic
        c_data["pmDE_norm"] = (c_data["pmDE"] - np.median(c_data["pmDE"])) / sigma_pm_intrinsic

        metadata[cluster_name] = {
            "C": C,
            "D_cluster": med_dist,
            "u_los": u_los,
            "e1": e1,
            "e2": e2,
            "D_trans": D_trans,
            "sigma_pm_intrinsic": sigma_pm_intrinsic,
        }
        cluster_results.append(c_data)

    if len(cluster_results) == 0:
        return pd.DataFrame(), {}
    out = pd.concat(cluster_results, ignore_index=True)

    return out, metadata

In [5]:
df_5d, geom_meta = prep_5d_astrometry(df)

In [6]:
good_clusters = []
for name, grp in df_5d.groupby("Name"):
    if len(grp) < 5:
        continue
    med_plx = np.median(grp["Plx"])
    med_dist = 1000 / med_plx
    if med_dist < 220:
        good_clusters.append(name)
df_near = df_5d[df_5d["Name"].isin(good_clusters)]

Klasterizavimas pagal spiečių formas

In [ ]:
feature_cols = ["X_norm", "Y_norm", "Z_norm", "pmRA_norm", "pmDE_norm"]
cluster_vectors = []
cluster_names = []
for name, grp in df_near.groupby("Name"):
    grp = grp.dropna(subset=feature_cols + ["Plx", "e_Plx"])
    if len(grp) < 20:
        continue
    X = grp[feature_cols].values
    cov = np.cov(X, rowvar=False)
    idx = np.triu_indices_from(cov)
    cov_features = cov[idx]
    cluster_vectors.append(cov_features)
    cluster_names.append(name)
morph_df = pd.DataFrame(cluster_vectors)
morph_df["Name"] = cluster_names
print("Clusters used:", len(morph_df))

X_morph = morph_df.drop(columns=["Name"]).values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_morph)
bic_scores = []
models = []
n_range = range(1, 10)
for n in n_range:
    gmm = GaussianMixture(
        n_components=n,
        covariance_type="full",
        random_state=42,
        n_init=10
    )
    gmm.fit(X_scaled)
    bic = gmm.bic(X_scaled)
    bic_scores.append(bic)
    models.append(gmm)
best_idx = np.argmin(bic_scores)
best_n = n_range[best_idx]
best_gmm = models[best_idx]
print(f"Best number of morphology groups: {best_n}")
labels = best_gmm.predict(X_scaled)
morph_df["Morph_Group"] = labels
morph_df["Morph_Prob"] = (best_gmm.predict_proba(X_scaled).max(axis=1))

print(morph_df[["Name", "Morph_Group", "Morph_Prob"]].sort_values("Morph_Group"))

In [ ]:
star_df = df_near.merge(morph_df[["Name", "Morph_Group"]], on="Name", how="inner")
features = ["X_norm", "Y_norm", "Z_norm", "pmRA_norm", "pmDE_norm"]
groups = sorted(star_df["Morph_Group"].unique())

for g in groups:
    grp = star_df[star_df["Morph_Group"] == g].copy()
    print(f"Group {g}: {len(grp)} stars")
    def corrfunc(x, y, **kws):
        r = np.corrcoef(x, y)[0, 1]
        ax = plt.gca()
        ax.annotate(f"{r:.2f}", xy=(0.5, 0.5), xycoords=ax.transAxes, ha="center",
                    va="center", fontsize=10)
        ax.set_axis_off()
    gplot = sns.PairGrid(grp[features], height=1.8)
    gplot.map_lower(sns.kdeplot, fill=True, levels=5, thresh=0.05)
    gplot.map_diag(sns.kdeplot, fill=True)
    gplot.map_upper(corrfunc)
    gplot.fig.suptitle(f"Group {g}", y=1.02, fontsize=14)
    plt.show()

Požymių kūrimas

In [9]:
def compute_cluster_features(star_df, include_shape_class=True):
    cluster_features = []
    for name, g in star_df.groupby("Name"):
        if len(g) < 20:
            continue
        eps = 1e-6
        x = g["X_norm"].values
        y = g["Y_norm"].values
        pmra = g["pmRA_norm"].values
        pmde = g["pmDE_norm"].values
        sx = np.std(x)
        sy = np.std(y)
        spmra = np.std(pmra)
        spmde = np.std(pmde)
        xy_ratio = sx / (sy + eps)
        pm_ratio = spmra / (spmde + eps)
        xy_corr = np.corrcoef(x, y)[0, 1]
        pm_corr = np.corrcoef(pmra, pmde)[0, 1]
        corr_x_pmra = np.corrcoef(x, pmra)[0, 1]
        corr_y_pmde = np.corrcoef(y, pmde)[0, 1]
        corr_x_pmde = np.corrcoef(x, pmde)[0, 1]
        corr_y_pmra = np.corrcoef(y, pmra)[0, 1]
        r_xy = np.sqrt(x**2 + y**2)
        pm_amp = np.sqrt(pmra**2 + pmde**2)
        corr_radius_pmamp = np.corrcoef(r_xy, pm_amp)[0, 1]
        cov_xy = np.cov(np.vstack([x, y]))
        eigvals = np.linalg.eigvalsh(cov_xy)
        eigvals = np.sort(eigvals)[::-1]
        planar_anisotropy = eigvals[0] / (eigvals[1] + eps)

        #kintamieji, kurie naudojami viename iš dviejų modelių 
        z = g["Z_norm"].values
        sz = np.std(z)
        z_ratio = sz / (0.5 * (sx + sy) + eps)
        corr_z_pmra = np.corrcoef(z, pmra)[0,1]
        corr_z_pmde = np.corrcoef(z, pmde)[0,1]
        corr_z_pmamp = np.corrcoef(z, pm_amp)[0,1]

        row = {
            "Name": name,
            "xy_ratio": xy_ratio,
            "planar_anisotropy": planar_anisotropy,
            "xy_corr": xy_corr,
            "pm_ratio": pm_ratio,
            "pm_corr": pm_corr,
            "corr_x_pmra": corr_x_pmra,
            "corr_y_pmde": corr_y_pmde,
            "corr_x_pmde": corr_x_pmde,
            "corr_y_pmra": corr_y_pmra,
            "corr_radius_pmamp": corr_radius_pmamp,
            
           # "z_ratio": z_ratio,
           # "corr_z_pmra": corr_z_pmra,
           # "corr_z_pmde": corr_z_pmde,
           # "corr_z_pmamp": corr_z_pmamp,
        }
        if include_shape_class:
            row["shape_class"] = g["Morph_Group"].mode().iloc[0]
        cluster_features.append(row)
    return pd.DataFrame(cluster_features)

cluster_df = compute_cluster_features(star_df)

In [ ]:
features = [
    "xy_ratio","xy_corr","planar_anisotropy","pm_ratio","pm_corr",
    "corr_x_pmra","corr_y_pmde","corr_x_pmde","corr_y_pmra",
    "corr_radius_pmamp", "z_ratio", "corr_z_pmra", "corr_z_pmde", "corr_z_pmamp"
]
corr = cluster_df[features].corr()
plt.figure(figsize=(14, 12))
cmap = sns.diverging_palette(280, 20, s=85, l=55, as_cmap=True)
sns.heatmap(corr, annot=True, fmt=".2f", cmap=cmap, center=0, square=True, 
            annot_kws={"size": 14}, cbar_kws={"format": lambda x, _: f"{x:.2f}".replace(".", ",")})
ax = plt.gca()
for text in ax.texts:
    text.set_text(text.get_text().replace(".", ","))
plt.xticks(fontsize=13, rotation=45, ha="right")
plt.yticks(fontsize=13)
plt.title("Požymių koreliacijų matrica", fontsize=18)
plt.tight_layout()
plt.show()

Formų klasifikavimas

In [ ]:
features = ["xy_ratio","xy_corr","planar_anisotropy","pm_ratio","pm_corr",
            "corr_x_pmra","corr_y_pmde","corr_x_pmde","corr_y_pmra","corr_radius_pmamp",
            # "z_ratio", "corr_z_pmra", "corr_z_pmde", "corr_z_pmamp"
            ]

X = cluster_df[features].copy()
y = cluster_df["shape_class"].copy()
mask = X.notna().all(axis=1) & y.notna()
X = X[mask]
y = y[mask]
le = LabelEncoder()
y_enc = le.fit_transform(y)
cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42
)

rf = RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1)

rf_params = {
    "n_estimators": randint(100, 500),
    "max_depth": [None, 3, 5, 7, 10],
    "min_samples_split": randint(2, 10),
    "min_samples_leaf": randint(1, 5),
    "max_features": ["sqrt", "log2", None],
    "bootstrap": [True, False],
}

rf_search = RandomizedSearchCV(
    rf,
    rf_params,
    n_iter=20,
    scoring="balanced_accuracy",
    cv=3,
    n_jobs=-1,
    random_state=42
)

svm = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf", class_weight="balanced"))
])

svm_params = {
    "clf__C": loguniform(1e-3, 1e2),
    "clf__gamma": loguniform(1e-4, 1e-1),
}

svm_search = RandomizedSearchCV(
    svm,
    svm_params,
    n_iter=20,
    scoring="balanced_accuracy",
    cv=3,
    n_jobs=-1,
    random_state=42
)

knn = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", KNeighborsClassifier())
])

knn_params = {
    "clf__n_neighbors": randint(3, 15),
    "clf__weights": ["uniform", "distance"],
    "clf__p": [1, 2],
}

knn_search = RandomizedSearchCV(
    knn,
    knn_params,
    n_iter=20,
    scoring="balanced_accuracy",
    cv=3,
    n_jobs=-1,
    random_state=42
)

logreg = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        multi_class="multinomial",
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    ))
])

logreg_params = {
    "clf__C": loguniform(1e-4, 1e2),
    "clf__penalty": ["l2"],
    "clf__solver": ["lbfgs", "saga"],
}

logreg_search = RandomizedSearchCV(
    logreg,
    logreg_params,
    n_iter=20,
    scoring="balanced_accuracy",
    cv=3,
    n_jobs=-1,
    random_state=42
)

searches = {
    "RandomForest": rf_search,
    "SVM": svm_search,
    "KNN": knn_search,
    "LogisticRegression": logreg_search,
}

results = []

for name, search in searches.items():
    search.fit(X, y_enc)
    best_model = search.best_estimator_
    scores = cross_validate(best_model, X, y_enc, cv=cv, scoring=["accuracy", "balanced_accuracy", "f1_macro"], n_jobs=-1)
    results.append({
        "model": name,
        "best_params": search.best_params_,
        "accuracy_mean": np.mean(scores["test_accuracy"]),
        "accuracy_std": np.std(scores["test_accuracy"]),
        "balanced_accuracy_mean": np.mean(scores["test_balanced_accuracy"]),
        "balanced_accuracy_std": np.std(scores["test_balanced_accuracy"]),
        "f1_macro_mean": np.mean(scores["test_f1_macro"]),
        "f1_macro_std": np.std(scores["test_f1_macro"]),
    })
results_df = pd.DataFrame(results)
results_df = results_df.sort_values("balanced_accuracy_mean", ascending=False)
print(results_df)

In [ ]:
best_rf = rf_search.best_estimator_
cv_cm = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_pred = cross_val_predict(best_rf, X, y_enc, cv=cv_cm, n_jobs=-1)
cm = confusion_matrix(y_enc, y_pred, normalize="true")
annot = np.vectorize(lambda x: f"{x:.2f}".replace(".", ","))(cm)
cmap = LinearSegmentedColormap.from_list("custom_purple", ["#f3eaff", "#8f5fca"])
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=annot, fmt="", cmap=cmap, xticklabels=np.array(le.classes_) + 1, yticklabels=np.array(le.classes_) + 1,
            cbar_kws={"label": "Teisingai prognozuota dalis"}, ax=ax, annot_kws={"size": 13})
cbar = ax.collections[0].colorbar
ticks = cbar.get_ticks()
cbar.set_ticklabels([f"{t:.2f}".replace(".", ",") for t in ticks])
ax.set_xlabel("Prognozuota forma")
ax.set_ylabel("Tikroji forma")
ax.set_title("Atsitiktinio miško modelio klasifikavimo matrica", size = 14)
plt.show()

Prognozė tolimiems

In [ ]:
far_clusters = []
med_dist_map = {}
for name, g in df_5d.groupby("Name"):
    med_plx = np.median(g["Plx"])
    med_dist = 1000.0 / med_plx
    med_dist_map[name] = med_dist
    if med_dist >= 220:
        far_clusters.append(name)
df_5d["med_dist"] = df_5d["Name"].map(med_dist_map)
far_df = df_5d[df_5d["Name"].isin(far_clusters)].copy()
print("Far clusters:", len(far_clusters))

In [ ]:
model = rf_search.best_estimator_
far_cluster_features = compute_cluster_features(far_df, include_shape_class=False)
far_cluster_features = far_cluster_features.merge(df_5d[["Name", "med_dist"]].drop_duplicates("Name"), on="Name", how="left")
features_rf = ["xy_ratio", "xy_corr", "planar_anisotropy", "pm_ratio", "pm_corr", "corr_x_pmra",
               "corr_y_pmde", "corr_x_pmde", "corr_y_pmra", "corr_radius_pmamp",]
X_far = far_cluster_features[features_rf].copy()
mask_far = X_far.notna().all(axis=1)
X_far = X_far[mask_far]
far_cluster_features = far_cluster_features.loc[mask_far].copy()
pred_enc = model.predict(X_far)
pred_class = le.inverse_transform(pred_enc)
pred_probs = model.predict_proba(X_far)
far_cluster_features["predicted_class"] = pred_class
for i, cls in enumerate(le.classes_):
    far_cluster_features[f"prob_class_{cls}"] = pred_probs[:, i]
print("\nPredicted morphology classes:")
print(far_cluster_features[["Name", "predicted_class"]].head(5))

In [ ]:
df_bins = far_cluster_features.copy()
bins = np.linspace(df_bins["med_dist"].min(), df_bins["med_dist"].max(), 20)
df_bins["dist_bin"] = pd.cut(df_bins["med_dist"], bins)
counts = df_bins.groupby(["dist_bin", "predicted_class"]).size().unstack(fill_value=0)
counts_norm = counts.div(counts.sum(axis=1), axis=0)
x = [interval.mid for interval in counts.index]
colors = {0: "#8f5fca", 1: "#62D7D7", 2: "#CB5928", 3: "#4580DF", 4: "#F4B400"}
def comma_formatter(x, pos):
    return f"{x:.1f}".replace(".", ",")
fig, ax = plt.subplots(figsize=(10, 6))
for cls in sorted(counts.columns):
    ax.plot(x, counts_norm[cls], marker='o', color=colors.get(cls, "black"), label=f"{cls+1} forma")
ax.set_xlabel("Spiečiaus atstumas, pc")
ax.set_ylabel("Spiečių dalis pagal prognozuotą formą")
ax.set_title("Spiečių formos prognozės priklausomybė nuo atstumo")
ax.yaxis.set_major_formatter(FuncFormatter(comma_formatter))
ax.legend()
ax.grid(True)
plt.show()

Korekcija

In [ ]:
def bootstrap_alpha(x, y, z, B=200):
    alphas = []
    n = len(x)
    for _ in range(B):
        idx = np.random.choice(n, n, replace=True)
        xb = x[idx]
        yb = y[idx]
        zb = z[idx]
        sigma_xy = 0.5 * (np.std(xb) + np.std(yb))
        sigma_z = np.std(zb)
        alpha_b = sigma_z / (sigma_xy + 1e-6)
        alphas.append(alpha_b)
    return np.array(alphas)

bootstrap_results = []
for name, g in star_df.groupby("Name"):
    if len(g) < 20:
        continue
    x = g["X_norm"].values
    y = g["Y_norm"].values
    z = g["Z_norm"].values
    alphas = bootstrap_alpha(x, y, z, B=300)
    morph_group = g["Morph_Group"].iloc[0]
    for a in alphas:
        bootstrap_results.append({"Morph_Group": morph_group, "alpha": a})
bootstrap_df = pd.DataFrame(bootstrap_results)
print(bootstrap_df.head())

In [ ]:
corrected_rows = []
for name, g in far_df.groupby("Name"):
    if name not in far_cluster_features["Name"].values:
        continue
    morph_class = (far_cluster_features.loc[far_cluster_features["Name"] == name, "predicted_class"].iloc[0])
    alpha_pool = bootstrap_df.loc[bootstrap_df["Morph_Group"] == morph_class, "alpha"].values
    alpha_prior_value = np.random.choice(alpha_pool)
    x = g["X_norm"].values
    y = g["Y_norm"].values
    z_obs = g["Z_norm"].values
    sigma_xy = 0.5 * (np.std(x) + np.std(y))
    sigma_z_obs = np.std(z_obs)
    sigma_z_target = alpha_prior_value * sigma_xy
    scale_factor = sigma_z_target / (sigma_z_obs + 1e-6)
    z_corr = z_obs * scale_factor
    tmp = g.copy()
    tmp["predicted_class"] = morph_class
    tmp["alpha_prior"] = alpha_prior_value
    tmp["sigma_z_target"] = sigma_z_target
    tmp["scale_factor"] = scale_factor
    tmp["Z_corr"] = z_corr
    corrected_rows.append(tmp)

corrected_far_df = pd.concat(corrected_rows, ignore_index=True)
print("\nRecovered far clusters:")
print(corrected_far_df.groupby("Name")[["alpha_prior", "scale_factor"]].mean().head(20))

In [17]:
from astroquery.vizier import Vizier
v = Vizier(columns=['Cluster', 'logAge50'], row_limit=-1)
cats = v.get_catalogs('J/AJ/167/12')
clusters = cats['J/AJ/167/12/catalog']
clusters = clusters.to_pandas()
final = corrected_far_df.merge(clusters, left_on="Name", right_on="Cluster", how="inner")
final = final.drop(columns=["Cluster"])

In [ ]:
def mean_interdistance(group):
    coords = group[['X_norm','Y_norm','Z_corr']].values
    if len(coords) < 2:
        return np.nan
    return pdist(coords).mean()
Di_per_cluster = final.groupby('Name').apply(mean_interdistance)
final['Di_corr'] = final['Name'].map(Di_per_cluster)

def mean_closest_interdistance(group):
    coords = group[['X_norm','Y_norm','Z_corr']].values
    n = len(coords)
    if n < 2:
        return np.nan
    dist_matrix = squareform(pdist(coords))
    np.fill_diagonal(dist_matrix, np.inf)
    min_dists = dist_matrix.min(axis=1)
    return min_dists.mean()
Dc_per_cluster = final.groupby('Name').apply(mean_closest_interdistance)
final['Dc_corr'] = final['Name'].map(Dc_per_cluster)

def mean_interdistance(group):
    coords = group[['X_norm','Y_norm','Z_norm']].values
    if len(coords) < 2:
        return np.nan
    return pdist(coords).mean()
Di_per_cluster = final.groupby('Name').apply(mean_interdistance)
final['Di_old'] = final['Name'].map(Di_per_cluster)

def mean_closest_interdistance(group):
    coords = group[['X_norm','Y_norm','Z_norm']].values
    n = len(coords)
    if n < 2:
        return np.nan
    dist_matrix = squareform(pdist(coords))
    np.fill_diagonal(dist_matrix, np.inf)
    min_dists = dist_matrix.min(axis=1)
    return min_dists.mean()
Dc_per_cluster = final.groupby('Name').apply(mean_closest_interdistance)
final['Dc_old'] = final['Name'].map(Dc_per_cluster)

def mean_interdistance(group):
    coords = group[['X_norm','Y_norm','Z_norm']].values
    if len(coords) < 2:
        return np.nan
    return pdist(coords).mean()
Di_per_cluster = df_near.groupby('Name').apply(mean_interdistance)
df_near['Di_near'] = df_near['Name'].map(Di_per_cluster)

def mean_closest_interdistance(group):
    coords = group[['X_norm','Y_norm','Z_norm']].values
    n = len(coords)
    if n < 2:
        return np.nan
    dist_matrix = squareform(pdist(coords))
    np.fill_diagonal(dist_matrix, np.inf)
    min_dists = dist_matrix.min(axis=1)
    return min_dists.mean()
Dc_per_cluster = df_near.groupby('Name').apply(mean_closest_interdistance)
df_near['Dc_near'] = df_near['Name'].map(Dc_per_cluster)

df_near = df_near.merge(clusters, left_on="Name", right_on="Cluster", how="inner")
df_near = df_near.drop(columns=["Cluster"])

In [ ]:
cluster_rows = []
for name, g in final.groupby("Name"):
    age_gyr = (10**g["logAge50"].iloc[0]) / 1e9
    cluster_rows.append({
        "Name": name,
        "Age_Gyr": age_gyr,
        "Di_corr": g["Di_corr"].iloc[0],
        "Dc_corr": g["Dc_corr"].iloc[0],
        "Di_old": g["Di_old"].iloc[0],
        "Dc_old": g["Dc_old"].iloc[0],
    })
clusters = pd.DataFrame(cluster_rows)

near_rows = []
for name, g in df_near.groupby("Name"):
    age_gyr = (10**g["logAge50"].iloc[0]) / 1e9
    near_rows.append({
        "Name": name,
        "Age_Gyr": age_gyr,
        "Di_near": g["Di_near"].iloc[0],
        "Dc_near": g["Dc_near"].iloc[0],
    })
near_clusters = pd.DataFrame(near_rows)

fig, axes = plt.subplots(2, 1, figsize=(7, 12), sharex=True)
new_cols = ["Di_corr", "Dc_corr"]
old_cols = ["Di_old", "Dc_old"]
near_cols = ["Di_near", "Dc_near"]
ytitles = ['Vidutinis atstumas tarp žvaigždžių $D_i$', 'Vidutinis artimiausias atstumas $D_c$']

def add_trend(ax, x, y, color, label):
    mask = (x > 0) & (y > 0)
    x = x[mask]
    y = y[mask]

    if len(x) < 2:
        return None, None, None
    logx = np.log10(x)
    logy = np.log10(y)
    m, b = np.polyfit(logx, logy, 1)
    xfit = np.linspace(logx.min(), logx.max(), 200)
    yfit = 10**(m * xfit + b)
    line, = ax.plot(10**xfit, yfit, linestyle="--", color=color, linewidth=2)
    m_str = f"{m:.2f}".replace(".", ",")
    return m, line, m_str

for ax, new_col, old_col, near_col, ytitle in zip(axes, new_cols, old_cols, near_cols, ytitles):
    tmp = clusters.dropna(subset=["Age_Gyr", new_col, old_col]).copy()
    tmp_near = near_clusters.dropna(subset=["Age_Gyr", near_col]).copy()
    x = tmp["Age_Gyr"].values
    y_new = tmp[new_col].values
    y_old = tmp[old_col].values
    x_near = tmp_near["Age_Gyr"].values
    y_near = tmp_near[near_col].values
    ax.scatter(x, y_old, s=25, alpha=0.7, label="Stebimi", color="#8f5fca")
    ax.scatter(x, y_new, s=25, alpha=0.7, label="Koreguoti", color="#62D7D7")
    ax.scatter(x_near, y_near, s=25, alpha=0.7, label="Artimi", color="#CB5928")
    m1, l1, m1s = add_trend(ax, x, y_old, "#7a44bc", "Stebimų")
    m2, l2, m2s = add_trend(ax, x, y_new, "#21BDBD", "Koreguotų")
    m3, l3, m3s = add_trend(ax, x_near, y_near, "#BE440F", "Artimų")
    handles, labels = ax.get_legend_handles_labels()
    handles += [
        l1, l2, l3
    ]
    labels += [
        f"Stebimų tendencija (k = {m1s})",
        f"Koreguotų tendencija (k = {m2s})",
        f"Artimų tendencija (k = {m3s})"
    ]
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_ylabel(ytitle)
    ax.legend(handles, labels, fontsize=9)

axes[-1].set_xlabel("Amžius, Gyr")
plt.suptitle("Žvaigždžių spiečių erdvinės metrikos priklausomybė nuo amžiaus", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
features = ["X_norm", "Y_norm", "Z_corr", "pmRA_norm", "pmDE_norm"]
groups = sorted(corrected_far_df["predicted_class"].unique())
main_color = "#62D7D7"
comma_formatter = FuncFormatter(lambda x, pos: f"{x:.1f}".replace(".", ","))

for g in groups:
    grp = corrected_far_df[corrected_far_df["predicted_class"] == g].copy()
    print(f"Group {g}: {len(grp)} stars, {grp['Name'].nunique()} clusters")

    def corrfunc(x, y, **kws):
        r = np.corrcoef(x, y)[0, 1]
        ax = plt.gca()
        r_text = f"{r:.2f}".replace(".", ",")
        ax.annotate(r_text, xy=(0.5, 0.5), xycoords=ax.transAxes, ha="center",
                    va="center", fontsize=11)
        ax.set_xticklabels([])
        ax.set_yticklabels([])
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_color("black")
            spine.set_linewidth(1)

    gplot = sns.PairGrid(grp[features], height=1.8)
    gplot.map_lower(sns.kdeplot, fill=True, levels=5, thresh=0.05, color=main_color)
    gplot.map_diag(sns.kdeplot, fill=True, color=main_color)
    gplot.map_upper(corrfunc)
    for ax_row in gplot.axes:
        for ax in ax_row:
            if ax is not None:
                ax.xaxis.set_major_formatter(comma_formatter)
                ax.yaxis.set_major_formatter(comma_formatter)
                for spine in ax.spines.values():
                    spine.set_visible(True)
                    spine.set_color("black")
                    spine.set_linewidth(1)

    gplot.fig.suptitle(f"{g+1} grupė pagal spiečių formą", y=0.98, fontsize=14)
    plt.tight_layout()
    plt.show()